In [4]:
import pandas as pd
import numpy as np
import plotly.express as px

from pathlib import Path

ROOT = Path.cwd().parent
print(ROOT)

DATA_RAW = ROOT/"data/raw"
DATA_PROCESSED = ROOT/"data/processed"

c:\Users\sebas\PycharmProjects\Git\BoxOffice_Oracle


In [5]:
model_df = pd.read_csv(
    DATA_RAW/"default_model_df.csv"
)

print(model_df.shape)
model_df.head()

(2255, 47)


,Unnamed: 0,tconst,primaryTitle,startYear,the_numbers_url,scrape_success,scrape_error,opening_weekend_gross,opening_theaters,domestic_release_date,...,actor_2,actor_3,actor_4,actor_5,actor_6,director_name,writer_name,actor_1_name,actor_2_name,actor_3_name
0,0,tt0120667,Fantastic Four,2005.0,https://www.the-numbers.com/movie/Fantastic-Fo...,True,NaN,56061504.0,3602.0,2005-07-08,...,nm0004821,nm0262635,nm0004695,nm0573037,nm0512934,Tim Story,Mark Frost,Ioan Gruffudd,Michael Chiklis,Chris Evans
1,1,tt0121164,Corpse Bride,2005.0,https://www.the-numbers.com/movie/Corpse-Bride,True,NaN,19145480.0,3204.0,2005-09-16,...,nm0000307,nm0001833,nm0001808,nm0925768,NaN,Tim Burton,John August,Johnny Depp,Helena Bonham Carter,Emily Watson
2,2,tt0121766,Star Wars: Episode III - Revenge of the Sith,2005.0,https://www.the-numbers.com/movie/Star-Wars-Ep...,True,NaN,108435841.0,3661.0,2005-05-19,...,nm0000204,nm0000191,nm0000168,nm0001519,nm0001751,George Lucas,George Lucas,Hayden Christensen,Natalie Portman,Ewan McGregor
3,3,tt0167190,Hellboy,2004.0,https://www.the-numbers.com/movie/Hellboy,True,NaN,23172440.0,3028.0,2004-04-02,...,nm0427964,nm0004757,nm0000457,nm1140344,nm0734558,Guillermo del Toro,Guillermo del Toro,Ron Perlman,Doug Jones,Selma Blair
4,4,tt0200465,The Bank Job,2008.0,https://www.the-numbers.com/movie/Bank-Job-The,True,NaN,5935256.0,1603.0,2008-03-07,...,nm0004787,nm1304386,nm0990547,nm0269077,nm0202810,Roger Donaldson,Dick Clement,Jason Statham,Saffron Burrows,Stephen Campbell Moore


In [6]:
model_df.columns

Index(['Unnamed: 0', 'tconst', 'primaryTitle', 'startYear', 'the_numbers_url',
       'scrape_success', 'scrape_error', 'opening_weekend_gross',
       'opening_theaters', 'domestic_release_date', 'release_type',
       'all_domestic_release_types', 'distributor', 'production_budget',
       'MPA_rating', 'MPA_rating_reason', 'runtime_minutes', 'genre',
       'subgenre', 'target_audience', 'time_period_setting', 'source',
       'production_method', 'creative_type', 'production_countries',
       'languages', 'legs', 'plot_point', 'raw_opening_weekend_text',
       'raw_theater_counts_text', 'raw_domestic_releases_text',
       'log_opening_weekend_gross', 'release_month', 'release_day_of_year',
       'director_id', 'writer_id', 'actor_1', 'actor_2', 'actor_3', 'actor_4',
       'actor_5', 'actor_6', 'director_name', 'writer_name', 'actor_1_name',
       'actor_2_name', 'actor_3_name'],
      dtype='str')

In [7]:
def add_group1_prior_genre_experience(model_df):
    df = model_df.copy()

    # Make sure dates/years sort correctly
    df["domestic_release_date"] = pd.to_datetime(df["domestic_release_date"], errors="coerce")
    df["startYear"] = pd.to_numeric(df["startYear"], errors="coerce")

    # Stable chronological order
    df = df.sort_values(
        ["domestic_release_date", "startYear", "primaryTitle"],
        na_position="last"
    ).reset_index(drop=True)

    # Entity columns to test
    entity_cols = {
        "director": "director_id",
        "writer": "writer_id",
        "actor_1": "actor_1",
        "actor_2": "actor_2",
        "actor_3": "actor_3",
        "distributor": "distributor"
    }

    # Prior films by entity within same genre
    for label, col in entity_cols.items():
        feature_name = f"g1_prior_same_genre_{label}_count"

        df[feature_name] = (
            df.groupby([col, "genre"])
              .cumcount()
        )

        # If entity is missing, set to 0
        df.loc[df[col].isna(), feature_name] = 0

    # Combined actor feature: total prior same-genre experience across top 3 actors
    actor_features = [
        "g1_prior_same_genre_actor_1_count",
        "g1_prior_same_genre_actor_2_count",
        "g1_prior_same_genre_actor_3_count"
    ]

    df["g1_prior_same_genre_top3_actor_total"] = df[actor_features].sum(axis=1)
    df["g1_prior_same_genre_top3_actor_max"] = df[actor_features].max(axis=1)
    df["g1_prior_same_genre_top3_actor_mean"] = df[actor_features].mean(axis=1)

    return df

In [8]:
model_df_g1 = add_group1_prior_genre_experience(model_df)

g1_features = [col for col in model_df_g1.columns if col.startswith("g1_")]

g1_features

['g1_prior_same_genre_director_count',
 'g1_prior_same_genre_writer_count',
 'g1_prior_same_genre_actor_1_count',
 'g1_prior_same_genre_actor_2_count',
 'g1_prior_same_genre_actor_3_count',
 'g1_prior_same_genre_distributor_count',
 'g1_prior_same_genre_top3_actor_total',
 'g1_prior_same_genre_top3_actor_max',
 'g1_prior_same_genre_top3_actor_mean']

In [10]:
model_df_g1.to_csv(DATA_RAW/"fe_groups/g1.csv")

## Group 2
### 3. Historical Momentum Features
- Actor, director, writer, studio, distributor, or genre gross revenue acceleration.
- Possible versions:
  - revenue velocity
  - change in revenue velocity
  - rolling historical growth rate

In [11]:
def add_group2_historical_momentum(model_df):
    df = model_df.copy()

    df["domestic_release_date"] = pd.to_datetime(df["domestic_release_date"], errors="coerce")
    df["opening_weekend_gross"] = pd.to_numeric(df["opening_weekend_gross"], errors="coerce")

    df = df.sort_values(
        ["domestic_release_date", "startYear", "primaryTitle"],
        na_position="last"
    ).reset_index(drop=True)

    entity_cols = {
        "director": "director_id",
        "writer": "writer_id",
        "actor_1": "actor_1",
        "actor_2": "actor_2",
        "actor_3": "actor_3",
        "distributor": "distributor",
        "genre": "genre"
    }

    for label, col in entity_cols.items():

        # Previous movie gross for that entity
        df[f"g2_prev_gross_{label}"] = (
            df.groupby(col)["opening_weekend_gross"]
              .shift(1)
        )

        # Rolling average of past 3 movies
        df[f"g2_rolling3_avg_gross_{label}"] = (
            df.groupby(col)["opening_weekend_gross"]
              .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
        )

        # Revenue velocity: average change between recent past movies
        df[f"g2_revenue_velocity_{label}"] = (
            df.groupby(col)["opening_weekend_gross"]
              .transform(lambda x: x.shift(1).diff().rolling(3, min_periods=1).mean())
        )

        # Acceleration: change in velocity
        df[f"g2_revenue_acceleration_{label}"] = (
            df.groupby(col)[f"g2_revenue_velocity_{label}"]
              .diff()
        )

        # Growth rate from previous historical film to current historical rolling average
        df[f"g2_rolling3_growth_rate_{label}"] = (
            df.groupby(col)["opening_weekend_gross"]
              .transform(lambda x: x.shift(1).pct_change().rolling(3, min_periods=1).mean())
        )

        # Missing entities should not create fake signal
        df.loc[df[col].isna(), [
            f"g2_prev_gross_{label}",
            f"g2_rolling3_avg_gross_{label}",
            f"g2_revenue_velocity_{label}",
            f"g2_revenue_acceleration_{label}",
            f"g2_rolling3_growth_rate_{label}"
        ]] = np.nan

    actor_momentum_cols = [
        "g2_rolling3_avg_gross_actor_1",
        "g2_rolling3_avg_gross_actor_2",
        "g2_rolling3_avg_gross_actor_3"
    ]

    df["g2_top3_actor_rolling3_avg_gross_mean"] = df[actor_momentum_cols].mean(axis=1)
    df["g2_top3_actor_rolling3_avg_gross_max"] = df[actor_momentum_cols].max(axis=1)
    df["g2_top3_actor_rolling3_avg_gross_total"] = df[actor_momentum_cols].sum(axis=1)

    return df

In [12]:
model_df_g2 = add_group2_historical_momentum(model_df)

g2_features = [col for col in model_df_g2.columns if col.startswith("g2_")]
g2_features

['g2_prev_gross_director',
 'g2_rolling3_avg_gross_director',
 'g2_revenue_velocity_director',
 'g2_revenue_acceleration_director',
 'g2_rolling3_growth_rate_director',
 'g2_prev_gross_writer',
 'g2_rolling3_avg_gross_writer',
 'g2_revenue_velocity_writer',
 'g2_revenue_acceleration_writer',
 'g2_rolling3_growth_rate_writer',
 'g2_prev_gross_actor_1',
 'g2_rolling3_avg_gross_actor_1',
 'g2_revenue_velocity_actor_1',
 'g2_revenue_acceleration_actor_1',
 'g2_rolling3_growth_rate_actor_1',
 'g2_prev_gross_actor_2',
 'g2_rolling3_avg_gross_actor_2',
 'g2_revenue_velocity_actor_2',
 'g2_revenue_acceleration_actor_2',
 'g2_rolling3_growth_rate_actor_2',
 'g2_prev_gross_actor_3',
 'g2_rolling3_avg_gross_actor_3',
 'g2_revenue_velocity_actor_3',
 'g2_revenue_acceleration_actor_3',
 'g2_rolling3_growth_rate_actor_3',
 'g2_prev_gross_distributor',
 'g2_rolling3_avg_gross_distributor',
 'g2_revenue_velocity_distributor',
 'g2_revenue_acceleration_distributor',
 'g2_rolling3_growth_rate_distributo

In [14]:
model_df_g2.to_csv(DATA_RAW/"fe_groups/g2.csv")

## Group 3
### Release Timing and Competition Features
- Distance from nearest seasonal demand peak.
- Nearest seasonal demand peak category.
- Number of similar-genre movies released within 20 days.
- Number of general movies released within 20 days.
- Number of similar-genre movies with similar budgets released within 20 days.
- Number of general movies with similar budgets released within 20 days.
- Budget × distance from nearest seasonal peak interaction.

In [16]:
def add_group3_release_timing_competition(model_df, window_days=20, budget_tolerance=0.30):
    df = model_df.copy()

    df["domestic_release_date"] = pd.to_datetime(df["domestic_release_date"], errors="coerce")
    df["production_budget"] = pd.to_numeric(df["production_budget"], errors="coerce")
    df["release_day_of_year"] = pd.to_numeric(df["release_day_of_year"], errors="coerce")

    df = df.sort_values("domestic_release_date").reset_index(drop=True)

    # Approximate seasonal demand peaks
    seasonal_peaks = {
        "summer": 185,        # early July
        "thanksgiving": 330,  # late November
        "christmas": 355,     # late December
        "spring": 90,         # late March / early April
        "halloween": 300      # late October
    }

    def nearest_peak(day):
        if pd.isna(day):
            return pd.Series([np.nan, np.nan])

        distances = {
            peak: min(abs(day - peak_day), 365 - abs(day - peak_day))
            for peak, peak_day in seasonal_peaks.items()
        }

        nearest = min(distances, key=distances.get)
        return pd.Series([distances[nearest], nearest])

    df[["g3_distance_from_nearest_seasonal_peak", "g3_nearest_seasonal_peak_category"]] = (
        df["release_day_of_year"].apply(nearest_peak)
    )

    # Competition features
    general_counts = []
    similar_genre_counts = []
    general_similar_budget_counts = []
    similar_genre_similar_budget_counts = []

    for idx, row in df.iterrows():
        release_date = row["domestic_release_date"]
        genre = row["genre"]
        budget = row["production_budget"]

        if pd.isna(release_date):
            general_counts.append(np.nan)
            similar_genre_counts.append(np.nan)
            general_similar_budget_counts.append(np.nan)
            similar_genre_similar_budget_counts.append(np.nan)
            continue

        start_date = release_date - pd.Timedelta(days=window_days)
        end_date = release_date + pd.Timedelta(days=window_days)

        nearby = df[
            (df["domestic_release_date"] >= start_date) &
            (df["domestic_release_date"] <= end_date) &
            (df.index != idx)
        ]

        general_counts.append(len(nearby))

        similar_genre = nearby[nearby["genre"] == genre]
        similar_genre_counts.append(len(similar_genre))

        if pd.isna(budget) or budget <= 0:
            general_similar_budget_counts.append(np.nan)
            similar_genre_similar_budget_counts.append(np.nan)
        else:
            lower_budget = budget * (1 - budget_tolerance)
            upper_budget = budget * (1 + budget_tolerance)

            nearby_similar_budget = nearby[
                (nearby["production_budget"] >= lower_budget) &
                (nearby["production_budget"] <= upper_budget)
            ]

            general_similar_budget_counts.append(len(nearby_similar_budget))

            similar_genre_similar_budget_counts.append(
                len(nearby_similar_budget[nearby_similar_budget["genre"] == genre])
            )

    df["g3_nearby_general_movies_20d_count"] = general_counts
    df["g3_nearby_similar_genre_movies_20d_count"] = similar_genre_counts
    df["g3_nearby_general_similar_budget_movies_20d_count"] = general_similar_budget_counts
    df["g3_nearby_similar_genre_similar_budget_movies_20d_count"] = similar_genre_similar_budget_counts

    # Interaction feature
    df["g3_budget_x_distance_from_peak"] = (
        df["production_budget"] * df["g3_distance_from_nearest_seasonal_peak"]
    )

    return df

In [17]:
model_df_g3 = add_group3_release_timing_competition(model_df)

g3_features = [col for col in model_df_g3.columns if col.startswith("g3_")]
g3_features

['g3_distance_from_nearest_seasonal_peak',
 'g3_nearest_seasonal_peak_category',
 'g3_nearby_general_movies_20d_count',
 'g3_nearby_similar_genre_movies_20d_count',
 'g3_nearby_general_similar_budget_movies_20d_count',
 'g3_nearby_similar_genre_similar_budget_movies_20d_count',
 'g3_budget_x_distance_from_peak']

In [18]:
model_df_g3.to_csv(DATA_RAW/"fe_groups/g3.csv")

## Group 4
### Local Genre Environment
- Mode genre among movies released within 60 days of the target movie.


In [19]:
def add_group4_local_genre_environment(model_df, window_days=60):
    df = model_df.copy()

    df["domestic_release_date"] = pd.to_datetime(df["domestic_release_date"], errors="coerce")

    df = df.sort_values("domestic_release_date").reset_index(drop=True)

    mode_genres = []
    mode_genre_counts = []

    for idx, row in df.iterrows():
        release_date = row["domestic_release_date"]

        if pd.isna(release_date):
            mode_genres.append(np.nan)
            mode_genre_counts.append(np.nan)
            continue

        start_date = release_date - pd.Timedelta(days=window_days)
        end_date = release_date + pd.Timedelta(days=window_days)

        nearby = df[
            (df["domestic_release_date"] >= start_date) &
            (df["domestic_release_date"] <= end_date) &
            (df.index != idx)
        ]

        if nearby.empty or nearby["genre"].dropna().empty:
            mode_genres.append(np.nan)
            mode_genre_counts.append(0)
        else:
            counts = nearby["genre"].value_counts()
            mode_genres.append(counts.index[0])
            mode_genre_counts.append(counts.iloc[0])

    df["g4_local_60d_mode_genre"] = mode_genres
    df["g4_local_60d_mode_genre_count"] = mode_genre_counts

    # Whether this movie matches the local dominant genre
    df["g4_matches_local_60d_mode_genre"] = (
        df["genre"] == df["g4_local_60d_mode_genre"]
    ).astype(int)

    return df

In [20]:
model_df_g4 = add_group4_local_genre_environment(model_df)

g4_features = [col for col in model_df_g4.columns if col.startswith("g4_")]
g4_features

['g4_local_60d_mode_genre',
 'g4_local_60d_mode_genre_count',
 'g4_matches_local_60d_mode_genre']

In [21]:
model_df_g4.to_csv(DATA_RAW/"fe_groups/g4.csv")

## Group 5
### Recency Features
- Days since actor’s last movie release.
- Days since director’s last movie release.

In [23]:
def add_group5_recency_features(model_df):
    df = model_df.copy()

    df["domestic_release_date"] = pd.to_datetime(df["domestic_release_date"], errors="coerce")

    df = df.sort_values(
        ["domestic_release_date", "startYear", "primaryTitle"],
        na_position="last"
    ).reset_index(drop=True)

    entity_cols = {
        "director": "director_id",
        "actor_1": "actor_1",
        "actor_2": "actor_2",
        "actor_3": "actor_3",
        "actor_4": "actor_4",
        "actor_5": "actor_5",
        "actor_6": "actor_6",
    }

    for label, col in entity_cols.items():
        last_release = df.groupby(col)["domestic_release_date"].shift(1)

        df[f"g5_days_since_last_release_{label}"] = (
            df["domestic_release_date"] - last_release
        ).dt.days

        df.loc[df[col].isna(), f"g5_days_since_last_release_{label}"] = np.nan

    actor_recency_cols = [
        "g5_days_since_last_release_actor_1",
        "g5_days_since_last_release_actor_2",
        "g5_days_since_last_release_actor_3",
        "g5_days_since_last_release_actor_4",
        "g5_days_since_last_release_actor_5",
        "g5_days_since_last_release_actor_6",
    ]

    df["g5_top6_actor_days_since_last_release_min"] = df[actor_recency_cols].min(axis=1)
    df["g5_top6_actor_days_since_last_release_mean"] = df[actor_recency_cols].mean(axis=1)
    df["g5_top6_actor_days_since_last_release_max"] = df[actor_recency_cols].max(axis=1)

    return df

In [24]:
model_df_g5 = add_group5_recency_features(model_df)

g5_features = [col for col in model_df_g5.columns if col.startswith("g5_")]
g5_features

['g5_days_since_last_release_director',
 'g5_days_since_last_release_actor_1',
 'g5_days_since_last_release_actor_2',
 'g5_days_since_last_release_actor_3',
 'g5_days_since_last_release_actor_4',
 'g5_days_since_last_release_actor_5',
 'g5_days_since_last_release_actor_6',
 'g5_top6_actor_days_since_last_release_min',
 'g5_top6_actor_days_since_last_release_mean',
 'g5_top6_actor_days_since_last_release_max']

In [25]:
model_df_g5.to_csv(DATA_RAW/"fe_groups/g5.csv")

## Group 6
### Relative Budget Competition Features



In [26]:
def add_group6_relative_budget_features(model_df, window_days=20):
    df = model_df.copy()

    df["domestic_release_date"] = pd.to_datetime(
        df["domestic_release_date"],
        errors="coerce"
    )

    df["production_budget"] = pd.to_numeric(
        df["production_budget"],
        errors="coerce"
    )

    df = df.sort_values("domestic_release_date").reset_index(drop=True)

    local_avg_budget = []
    local_median_budget = []
    local_max_budget = []
    num_larger_budget_competitors = []
    budget_percentile_local = []

    for idx, row in df.iterrows():

        release_date = row["domestic_release_date"]
        budget = row["production_budget"]

        if pd.isna(release_date) or pd.isna(budget):
            local_avg_budget.append(np.nan)
            local_median_budget.append(np.nan)
            local_max_budget.append(np.nan)
            num_larger_budget_competitors.append(np.nan)
            budget_percentile_local.append(np.nan)
            continue

        start_date = release_date - pd.Timedelta(days=window_days)
        end_date = release_date + pd.Timedelta(days=window_days)

        nearby = df[
            (df["domestic_release_date"] >= start_date) &
            (df["domestic_release_date"] <= end_date) &
            (df.index != idx)
        ]

        nearby_budgets = nearby["production_budget"].dropna()

        if len(nearby_budgets) == 0:
            local_avg_budget.append(np.nan)
            local_median_budget.append(np.nan)
            local_max_budget.append(np.nan)
            num_larger_budget_competitors.append(0)
            budget_percentile_local.append(np.nan)
            continue

        local_avg_budget.append(nearby_budgets.mean())
        local_median_budget.append(nearby_budgets.median())
        local_max_budget.append(nearby_budgets.max())

        num_larger_budget_competitors.append(
            (nearby_budgets > budget).sum()
        )

        percentile = (
            (nearby_budgets < budget).sum()
            / len(nearby_budgets)
        )

        budget_percentile_local.append(percentile)

    df["g6_local_avg_budget_20d"] = local_avg_budget
    df["g6_local_median_budget_20d"] = local_median_budget
    df["g6_local_max_budget_20d"] = local_max_budget

    df["g6_num_larger_budget_competitors_20d"] = (
        num_larger_budget_competitors
    )

    df["g6_budget_percentile_local_20d"] = (
        budget_percentile_local
    )

    # Relative budget ratios
    df["g6_budget_vs_local_avg_ratio"] = (
        df["production_budget"] /
        df["g6_local_avg_budget_20d"]
    )

    df["g6_budget_vs_local_median_ratio"] = (
        df["production_budget"] /
        df["g6_local_median_budget_20d"]
    )

    return df

In [27]:
model_df_g6 = add_group6_relative_budget_features(model_df)

g6_features = [col for col in model_df_g6.columns if col.startswith("g6_")]
g6_features

['g6_local_avg_budget_20d',
 'g6_local_median_budget_20d',
 'g6_local_max_budget_20d',
 'g6_num_larger_budget_competitors_20d',
 'g6_budget_percentile_local_20d',
 'g6_budget_vs_local_avg_ratio',
 'g6_budget_vs_local_median_ratio']

In [28]:
model_df_g6.to_csv(DATA_RAW/"fe_groups/g6.csv")